# CSOT: 约束分离最优传输相位恢复 (Julia)

## 约束分离 = 像素随机分组 + 相邻像素不能同组 → 非周期、有间隔的最优分布

### 集成方法
1. 将 `ConstraintSeparatedOT.jl` 复制到 `SLMTools-main/src/PhaseRetrieval/`
2. 在 `Project.toml` 的 `[deps]` 中添加 `Random`
3. 在 `SLMTools.jl` 中添加 `include` + `using` + `export`

本 notebook 直接用 `include` 加载模块进行演示。

In [ ]:
using SLMTools
using Plots, Images, FFTW, LinearAlgebra, Random
using SLMTools: wrap
import Interpolations: Linear

# 加载约束分离模块
include(raw"d:\6.2谈话资料\CSOT-improved\julia\ConstraintSeparatedOT.jl")
using .ConstraintSeparatedOT

println("✓ CSOT 模块加载成功")

## 1. 像素分组可视化 — 约束分离核心

展示 4色图着色：每个2×2块内4个不同组，相邻像素永远不同组。

In [ ]:
groups = createPixelGroups((16, 16), 4; seed=42)
info = verifyGroupConstraint(groups)

println("分组: $(info.n_groups)组 | 邻接对: $(info.adjacent_pairs) | 违反: $(info.violations) | 约束: $(info.valid ? "✓" : "✗")")
heatmap(groups, aspect_ratio=1, color=:tab10, title="像素分组 (相邻不同组)",
        xticks=0:15, yticks=0:15, size=(500, 500))

## 2. 生成测试数据 (128×128, 论文设置)

In [ ]:
N = 128
L0 = natlat((N, N))

# 输入光束: 高斯 σ=3 (匹配论文)
I_in = LF{ComplexAmp}(exp.(-r2(L0) ./ (3 + 0im)^2), L0)
I_in_intensity = square(abs(I_in)) |> normalizeLF

# 目标: 环形 (letter 'c' 替代)
R2_demo = [x^2 + y^2 for x in L0[1], y in L0[2]]
ring = exp.(-(sqrt.(R2_demo) .- 3.5) .^ 2 ./ (2 * 0.5^2))
I_target = LF{Intensity}(ring, L0, 1.0) |> normalizeLF

p1 = heatmap(I_in_intensity.data, title="源 (高斯 σ=3)", aspect_ratio=1, color=:hot)
p2 = heatmap(I_target.data, title="目标 (环形)", aspect_ratio=1, color=:hot)
plot(p1, p2, size=(800, 400))

## 3. 方法对比: 标准GS vs csotGS (约束分离)

使用相同的 OT 初始化，对比标准 GS 和约束分离分组 GS。

In [ ]:
# OT 初始化 (论文2方法)
Φ_ot = otPhase(I_in_intensity, I_target, 0.001)

# ---- 标准 GS (论文2) ----
println("标准 GS (全图同步, 100 iter)...")
Φ_gs = gs(I_in_intensity, I_target, 100, Φ_ot)
out_gs = normalizeLF(square(sft(sqrt(I_in_intensity) * Φ_gs)))
err_gs = sqrt(sum((out_gs.data - normalizeLF(I_target).data) .^ 2) / N^2)
println("  标准 GS RMS: $(round(err_gs, digits=6))")

# ---- 约束分离分组 GS (改进) ----
println("\n约束分离分组 GS (csotGS, 4组, 100 iter)...")
Φ_csot, groups, errors = csotGS(
    sqrt(I_in_intensity), sqrt(I_target), 100, Φ_ot;
    n_groups=4, verbose=true)
out_csot = normalizeLF(square(sft(sqrt(I_in_intensity) * Φ_csot)))
err_csot = sqrt(sum((out_csot.data - normalizeLF(I_target).data) .^ 2) / N^2)

println("\n=== 误差汇总 ===")
println("标准 GS  (100 iter, OT init):  RMS = $(round(err_gs, digits=6))")
println("csotGS   (100 iter, OT init):  RMS = $(round(err_csot, digits=6))")
println("改进: $(round((1 - err_csot/err_gs)*100, digits=1))%")

## 4. 完整 CSOT 流水线

`csotPhase` — OT初始化 + 约束分离精化，一步完成。

In [ ]:
Φ_final, Φ_ot_init, errors_full = csotPhase(
    I_in_intensity, I_target, 0.001;
    refine_iter=50, n_groups=4, verbose=true)

println("\n最终 RMS: $(round(errors_full[end], digits=6))")

## 5. 收敛曲线对比

`csotGSLog` — 记录每次迭代的误差，用于对比收敛速度。

In [ ]:
# 标准 GS 收敛
Φ_gs_log, errs_gs_log = gsLog(sqrt(I_in_intensity), sqrt(I_target), 200, Φ_ot; every=5)

# 约束分离分组 GS 收敛
Φ_csot_log, errs_csot_log = csotGSLog(
    sqrt(I_in_intensity), sqrt(I_target), 200, Φ_ot;
    n_groups=4, every=5)

# 绘制收敛曲线
iters = 0:5:199
plot(iters, [errs_gs_log errs_csot_log],
     label=["标准 GS" "csotGS (约束分离)"],
     xlabel="迭代次数", ylabel="GS误差",
     title="收敛曲线: 标准 GS vs 约束分离分组 GS",
     linewidth=2, yscale=:log10, legend=:topright,
     size=(800, 500))

## 算法原理

### 标准 GS vs 约束分离分组 GS

```
标准 gs():
  for it = 1:nit:
      guess = gsIter(guess, u, v, ft, ift)  # 全部像素同步更新

csotGS():
  groups = 4色图着色  # 相邻像素不同组
  for it = 1:nit:
      group_order = randperm(4)
      for g in group_order:
          mask = (groups == g)
          guess_local = gsIter(guess_local, u, v, ft, ift)
          phase[mask] .= phase_new[mask]  # ★ 仅更新本组!
```

### 论文复现

```julia
# 原来:
Φotgs = gs(I_in, I_target, 10000, wrap(Φot))

# 改为:
Φotcsot, _, _ = csotGS(sqrt(I_in), sqrt(I_target), 10000, Φot; n_groups=4)
```